# skill_master 新技能寫入 Notebook（合併版）

本 Notebook **合併**：
- `extract_tools_to_skill_master.ipynb` 的邏輯：從 `jobs_rows.csv` 的 `tools` 欄位萃取技能、統計出現次數、與既有 `skill_master` 對照。
- `skill_write_evaluation.md` 的評估準則：挑出建議新增的技能（Linux、C#、HTML、CSS、Shell、PyTorch、TensorFlow、R、Flutter、Android、iOS、jQuery、ASP.NET、Power BI、Tableau、ETL、Figma、Matlab、Sass、LLM、NLP、scikit-learn、JIRA、Vmware、RDBMS、PL/SQL、JSP、Visual Studio…）。

最終會直接**從 `jobs_rows.csv` 重新計算 count → 套用內建的評估表 → 寫入 Supabase 的 `skill_master` 表**，不再依賴外部 `skill_master_recommended_to_write.csv` 檔案。

**重點說明：**
- 僅寫入「評估為應該新增到 `skill_master`」的技能（內建 `action == "WRITE_NEW"` 的清單）。
- 嚴格遵守 ERD 中 `SKILL_MASTER` 欄位定義：只寫入 `skill_name`, `skill_category`, `synonyms`, `created_at`，不自行指定 `skill_id`（交由資料庫自動產生）。
- `synonyms` 以 JSON 陣列形式寫入（例如：`["linux", "LINUX"]`）。
- **只處理目前不在 `skill_master` 中的技能名稱**，避免重複寫入。

**前置條件：**
- 在 `supabase_control` 資料夾下存在：`jobs_rows.csv` 與 `skill_master_rows.csv`（前者來自職缺資料，後者可由現有 `skill_master` 匯出）。
- 已在執行環境中設定環境變數：`SUPABASE_URL`, `SUPABASE_KEY`，並確認 Supabase 專案中已建立 `skill_master` 資料表且欄位與 ERD 一致。

接下來的程式碼區塊將依步驟完成：
1. 匯入套件與載入環境變數、建立 Supabase 連線。
2. 從 `jobs_rows.csv` 重新萃取所有 tools 技能與出現次數、與 `skill_master_rows.csv` 對照出「目前尚未存在的技能」。
3. 在程式中依 `skill_write_evaluation.md` 內建一份「建議新增」清單並套用到上述候選技能。
4. 依 `count` 與評估結果挑出要寫入的技能，整理成 payload。
5. 依 `skill_name` 檢查資料庫是否已有紀錄，避免重複寫入，最後批次插入新的技能資料至 `skill_master`。

In [5]:
import os
import ast
from datetime import datetime, timezone

import pandas as pd
from dotenv import load_dotenv
from supabase import create_client

# 載入 .env（Notebook 不會自動載入，需手動呼叫）
load_dotenv()  # 從目前工作目錄載入 .env
if not os.environ.get("SUPABASE_URL"):
    load_dotenv(os.path.join(os.getcwd(), "supabase_control", ".env"))  # 若在專案根目錄執行

# === 參數設定 ===
MIN_COUNT = 100  # 第一波寫入：只處理 count >= 100 的技能

# Supabase 連線設定：從環境變數讀取（.env 已由 load_dotenv 載入）
SUPABASE_URL = os.environ.get("SUPABASE_URL")
SUPABASE_KEY = os.environ.get("SUPABASE_SERVICE_ROLE_KEY") or os.environ.get("SUPABASE_KEY")

assert SUPABASE_URL, "環境變數 SUPABASE_URL 尚未設定（請在 .env 設定或設定系統環境變數）"
assert SUPABASE_KEY, "環境變數 SUPABASE_SERVICE_ROLE_KEY 或 SUPABASE_KEY 尚未設定（請在 .env 設定）"

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

print("Supabase client 建立完成")

Supabase client 建立完成


In [6]:
# 簡單健康檢查：確認可以讀取 skill_master（只取 1 筆）
from supabase import Client

assert isinstance(supabase, Client)

health_check = supabase.table("skill_master").select("skill_id, skill_name").limit(1).execute()
print("health_check data:", health_check.data)
print("與 skill_master 連線正常（若此表目前為空，data 可能為 []）")

health_check data: [{'skill_id': 1, 'skill_name': 'Python'}]
與 skill_master 連線正常（若此表目前為空，data 可能為 []）


In [7]:
# 從 jobs_rows.csv 重新萃取技能，並套用內建評估規則，產生要寫入的技能清單

from collections import Counter

# === 3.1 讀取 jobs_rows 與 skill_master_rows，取得候選技能（尚未在 skill_master 的） ===

DATA_DIR = os.getcwd()
JOBS_CSV = os.path.join(DATA_DIR, "jobs_rows.csv")
SKILL_MASTER_CSV = os.path.join(DATA_DIR, "skill_master_rows.csv")

print("JOBS_CSV:", JOBS_CSV)
print("SKILL_MASTER_CSV:", SKILL_MASTER_CSV)

# 只讀必要欄位以節省記憶體
jobs = pd.read_csv(JOBS_CSV, usecols=["job_id", "job_name", "tools", "url"], encoding="utf-8", low_memory=False)
print(f"✓ 載入 {len(jobs):,} 筆 jobs")
print("tools 非空筆數:", jobs["tools"].notna().sum())


def split_tools(s: str):
    """將一筆 tools 字串拆成 list（支援全形頓號、半形逗號）。"""
    if pd.isna(s) or not str(s).strip():
        return []
    s = str(s).strip()
    # 統一轉成同一個分隔符再 split
    for sep in ["、", ","]:
        s = s.replace(sep, "|")
    return [x.strip() for x in s.split("|") if x.strip()]


all_tools = []
for v in jobs["tools"].dropna():
    all_tools.extend(split_tools(v))

print(f"拆出之技能總數（含重複）: {len(all_tools):,}")

counts = Counter(all_tools)
df_tools = (
    pd.DataFrame([{"skill_name": k, "count": v} for k, v in counts.items()])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
df_tools.index = df_tools.index + 1
df_tools.index.name = "index"
print(f"共 {len(df_tools):,} 個不重複技能（含已在 skill_master 的）")

# 載入目前 skill_master 匯出檔，標記哪些技能已存在
skill_master_df = pd.read_csv(SKILL_MASTER_CSV, encoding="utf-8")
existing_skills_lower = set(skill_master_df["skill_name"].astype(str).str.strip().str.lower())

df_tools["in_skill_master"] = df_tools["skill_name"].astype(str).str.strip().str.lower().isin(existing_skills_lower)

in_master = df_tools["in_skill_master"].sum()
not_in_master = (~df_tools["in_skill_master"]).sum()
print(f"在 jobs 的 tools 中：已在 skill_master 的 {in_master} 個，尚未在 skill_master 的 {not_in_master} 個")

# 僅保留「尚未在 skill_master」的候選技能（後續才會套用評估規則）
df_new_only = df_tools[~df_tools["in_skill_master"]].copy()
print("候選技能（尚未在 skill_master）前 30 筆：")
display(df_new_only.head(30))


# === 3.2 內建評估規則（摘自 skill_write_evaluation.md） ===

# key 一律用小寫 skill_name，比對時會 .lower().strip()
EVAL_RULES = {
    # skill_name: {action, skill_category, synonyms}
    "linux": {
        "action": "WRITE_NEW",
        "skill_category": "Tool",
        "synonyms": ["linux", "LINUX"],
    },
    "c#": {
        "action": "WRITE_NEW",
        "skill_category": "Programming",
        "synonyms": ["c#", "C#", "C Sharp"],
    },
    "html": {
        "action": "WRITE_NEW",
        "skill_category": "Programming",
        "synonyms": ["html", "HTML5"],
    },
    "css": {
        "action": "WRITE_NEW",
        "skill_category": "Programming",
        "synonyms": ["css", "CSS3"],
    },
    "c": {
        "action": "WRITE_NEW",
        "skill_category": "Programming",
        "synonyms": ["c", "C language"],
    },
    "excel": {
        "action": "WRITE_NEW",
        "skill_category": "Tool",
        "synonyms": ["excel", "Microsoft Excel"],
    },
    "shell": {
        "action": "WRITE_NEW",
        "skill_category": "Tool",
        "synonyms": ["shell", "bash", "Shell Script"],
    },
    "pytorch": {
        "action": "WRITE_NEW",
        "skill_category": "Tool",
        "synonyms": ["pytorch", "PyTorch"],
    },
    "tensorflow": {
        "action": "WRITE_NEW",
        "skill_category": "Tool",
        "synonyms": ["tensorflow", "tensor flow", "TensorFlow"],
    },
    "r": {
        "action": "WRITE_NEW",
        "skill_category": "Programming",
        "synonyms": ["r", "R language"],
    },
    "flutter": {
        "action": "WRITE_NEW",
        "skill_category": "Framework",
        "synonyms": ["flutter"],
    },
    "android": {
        "action": "WRITE_NEW",
        "skill_category": "Tool",
        "synonyms": ["android", "Android SDK"],
    },
    "ios": {
        "action": "WRITE_NEW",
        "skill_category": "Tool",
        "synonyms": ["ios", "iOS"],
    },
    "jquery": {
        "action": "WRITE_NEW",
        "skill_category": "Framework",
        "synonyms": ["jquery", "jQuery"],
    },
    "asp.net": {
        "action": "WRITE_NEW",
        "skill_category": "Framework",
        "synonyms": ["asp.net", "ASP.NET"],
    },
    "spring": {
        "action": "WRITE_NEW",
        "skill_category": "Framework",
        "synonyms": ["spring", "Spring Framework"],
    },
    "power bi": {
        "action": "WRITE_NEW",
        "skill_category": "Tool",
        "synonyms": ["power bi", "PowerBI"],
    },
    "tableau": {
        "action": "WRITE_NEW",
        "skill_category": "Tool",
        "synonyms": ["tableau"],
    },
    "etl": {
        "action": "WRITE_NEW",
        "skill_category": "Tool",
        "synonyms": ["etl"],
    },
    "figma": {
        "action": "WRITE_NEW",
        "skill_category": "Tool",
        "synonyms": ["figma"],
    },
    "matlab": {
        "action": "WRITE_NEW",
        "skill_category": "Tool",
        "synonyms": ["matlab", "MATLAB"],
    },
    "sass": {
        "action": "WRITE_NEW",
        "skill_category": "Programming",
        "synonyms": ["sass", "SCSS"],
    },
    "llm": {
        "action": "WRITE_NEW",
        "skill_category": "Tool",
        "synonyms": ["llm", "Large Language Model"],
    },
    "nlp": {
        "action": "WRITE_NEW",
        "skill_category": "Tool",
        "synonyms": ["nlp", "Natural Language Processing"],
    },
    "scikit-learn": {
        "action": "WRITE_NEW",
        "skill_category": "Tool",
        "synonyms": ["scikit-learn", "sklearn"],
    },
    "jira": {
        "action": "WRITE_NEW",
        "skill_category": "Tool",
        "synonyms": ["jira", "Jira"],
    },
    "vmware": {
        "action": "WRITE_NEW",
        "skill_category": "Tool",
        "synonyms": ["vmware", "VMware", "ESXi"],
    },
    "rdbms": {
        "action": "WRITE_NEW",
        "skill_category": "Tool",
        "synonyms": ["rdbms", "關聯式資料庫"],
    },
    "pl/sql": {
        "action": "WRITE_NEW",
        "skill_category": "Tool",
        "synonyms": ["pl/sql", "PLSQL"],
    },
    "jsp": {
        "action": "WRITE_NEW",
        "skill_category": "Framework",
        "synonyms": ["jsp", "Java Server Pages"],
    },
    "visual studio": {
        "action": "WRITE_NEW",
        "skill_category": "Tool",
        "synonyms": ["visual studio", "VS"],
    },
}


# === 3.3 套用評估規則，取得「建議新增(WRITE_NEW)」且符合 count 門檻的技能 ===

records = []
for _, row in df_new_only.iterrows():
    raw_name = str(row["skill_name"]).strip()
    key = raw_name.lower()
    rule = EVAL_RULES.get(key)
    if not rule:
        continue

    record = {
        "skill_name": raw_name,
        "count": int(row["count"]),
        "action": rule["action"],
        "skill_category": rule["skill_category"],
        "synonyms": rule["synonyms"],
    }
    records.append(record)

if not records:
    print("⚠ 依目前評估規則，df_new_only 中沒有對應的技能。請確認 jobs_rows 與 EVAL_RULES 是否一致。")
    filtered = pd.DataFrame(columns=["skill_name", "skill_category", "synonyms"])
else:
    df_eval = pd.DataFrame(records)
    print("建議清單摘要（依 action 分組）:")
    print(df_eval["action"].value_counts())

    # 只保留建議新增 (WRITE_NEW) 的技能，並套用 count 門檻
    mask_action = df_eval["action"].astype(str).str.upper() == "WRITE_NEW"
    mask_count = df_eval["count"] >= MIN_COUNT

    filtered = df_eval[mask_action & mask_count].copy()
    print("符合 WRITE_NEW 且 count >=", MIN_COUNT, "的列數:", len(filtered))

    # 僅保留與 SKILL_MASTER 對應的欄位
    # SKILL_MASTER: skill_name (UNIQUE, NOT NULL), skill_category, synonyms, created_at
    filtered = filtered[["skill_name", "skill_category", "synonyms"]]

print("=== 最終將作為寫入來源的 filtered 頭幾筆 ===")
filtered.head()

JOBS_CSV: c:\Users\Elvis\git\final\supabase_control\jobs_rows.csv
SKILL_MASTER_CSV: c:\Users\Elvis\git\final\supabase_control\skill_master_rows.csv
✓ 載入 18,729 筆 jobs
tools 非空筆數: 12657
拆出之技能總數（含重複）: 61,124
共 512 個不重複技能（含已在 skill_master 的）
在 jobs 的 tools 中：已在 skill_master 的 18 個，尚未在 skill_master 的 494 個
候選技能（尚未在 skill_master）前 30 筆：


,skill_name,count,in_skill_master
index,,,
3,C#,2641,False
6,HTML,2066,False
7,MS SQL,2045,False
8,Excel,2014,False
9,Linux,1950,False
10,CSS,1796,False
12,Word,1715,False
14,C,1633,False
15,PowerPoint,1508,False


建議清單摘要（依 action 分組）:
action
WRITE_NEW    31
Name: count, dtype: int64
符合 WRITE_NEW 且 count >= 100 的列數: 30
=== 最終將作為寫入來源的 filtered 頭幾筆 ===


,skill_name,skill_category,synonyms
0,C#,Programming,"[c#, C#, C Sharp]"
1,HTML,Programming,"[html, HTML5]"
2,Excel,Tool,"[excel, Microsoft Excel]"
3,Linux,Tool,"[linux, LINUX]"
4,CSS,Programming,"[css, CSS3]"


In [11]:
# 將 synonyms 欄位轉成真正的 Python list，並準備寫入 payload

def parse_synonyms(value):
    """將 synonyms 轉為 list，空值則回傳空 list。
    同時支援：
    - 已經是 list/tuple
    - 字串格式: "[\"linux\", \"LINUX\"]" 或 "linux;LINUX" 或 "linux,LINUX"
    """
    # 1) 若本來就是 list/tuple，直接回傳
    if isinstance(value, (list, tuple)):
        return list(value)

    # 2) 明確處理 None / NaN（只對 scalar 用 isna）
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except TypeError:
        # value 不是標量（例如 array），但前面已處理過 list/tuple，就先略過
        pass

    text = str(value).strip()
    if not text:
        return []

    # 3) 優先嘗試當作 Python/JSON list 解析
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, str):
            return [parsed]
    except Exception:
        pass

    # 4) 若不是 list 格式，嘗試用常見分隔符號切分
    if ";" in text:
        return [s.strip() for s in text.split(";") if s.strip()]
    if "," in text:
        return [s.strip() for s in text.split(",") if s.strip()]

    # 5) fallback：單一詞
    return [text]


filtered["synonyms_list"] = filtered["synonyms"].apply(parse_synonyms)

# 準備要寫入的基本欄位，不含 created_at（稍後統一補上）
records_to_prepare = [
    {
        "skill_name": row["skill_name"].strip(),
        "skill_category": str(row["skill_category"]).strip() if not pd.isna(row["skill_category"]) else None,
        "synonyms": row["synonyms_list"],
    }
    for _, row in filtered.iterrows()
]

len(records_to_prepare), records_to_prepare[:]

(30,
 [{'skill_name': 'C#',
   'skill_category': 'Programming',
   'synonyms': ['c#', 'C#', 'C Sharp']},
  {'skill_name': 'HTML',
   'skill_category': 'Programming',
   'synonyms': ['html', 'HTML5']},
  {'skill_name': 'Excel',
   'skill_category': 'Tool',
   'synonyms': ['excel', 'Microsoft Excel']},
  {'skill_name': 'Linux',
   'skill_category': 'Tool',
   'synonyms': ['linux', 'LINUX']},
  {'skill_name': 'CSS',
   'skill_category': 'Programming',
   'synonyms': ['css', 'CSS3']},
  {'skill_name': 'C',
   'skill_category': 'Programming',
   'synonyms': ['c', 'C language']},
  {'skill_name': 'ASP.NET',
   'skill_category': 'Framework',
   'synonyms': ['asp.net', 'ASP.NET']},
  {'skill_name': 'jQuery',
   'skill_category': 'Framework',
   'synonyms': ['jquery', 'jQuery']},
  {'skill_name': 'Visual Studio',
   'skill_category': 'Tool',
   'synonyms': ['visual studio', 'VS']},
  {'skill_name': 'Spring',
   'skill_category': 'Framework',
   'synonyms': ['spring', 'Spring Framework']},
  {'s

In [10]:
# 從資料庫讀取既有 skill_master，避免重複寫入

# 嚴格依 ERD：不指定 skill_id，由資料庫自動產生；
# 這裡僅用 skill_name (UNIQUE) 來判斷是否已存在。

existing = supabase.table("skill_master").select("skill_name").execute()
existing_names = {
    (row.get("skill_name") or "").strip().lower()
    for row in (existing.data or [])
}
print("資料庫中既有 skill_master 筆數:", len(existing_names))

# 過濾掉已存在的 skill_name
utc_now = datetime.now(timezone.utc).isoformat()

payload = []
for rec in records_to_prepare:
    name_key = rec["skill_name"].strip().lower()
    if not name_key:
        continue
    if name_key in existing_names:
        # 已存在，略過（如需改為 upsert，可在此調整策略）
        continue

    payload.append(
        {
            "skill_name": rec["skill_name"],
            "skill_category": rec["skill_category"],
            "synonyms": rec["synonyms"],  # JSON 陣列
            "created_at": utc_now,          # 由程式設定建立時間（UTC ISO 格式）
        }
    )

print("準備寫入的新技能筆數:", len(payload))
len(payload)

資料庫中既有 skill_master 筆數: 40
準備寫入的新技能筆數: 30


30

In [12]:
# 批次寫入 skill_master

if not payload:
    print("沒有新的技能需要寫入，請檢查 CSV 或 MIN_COUNT 設定。")
else:
    BATCH_SIZE = 50
    inserted_total = 0
    for i in range(0, len(payload), BATCH_SIZE):
        batch = payload[i : i + BATCH_SIZE]
        print(f"寫入批次 {i} ~ {i + len(batch) - 1} ...")
        resp = supabase.table("skill_master").insert(batch).execute()
        if resp.data is not None:
            inserted_total += len(resp.data)
        print("  本批次實際寫入筆數:", len(resp.data) if resp.data is not None else "未知")

    print("=== 寫入完成 ===")
    print("總預期寫入筆數:", len(payload))
    print("總實際寫入筆數:", inserted_total)

# 依需要可再補一個 select 檢查最終筆數

寫入批次 0 ~ 29 ...
  本批次實際寫入筆數: 30
=== 寫入完成 ===
總預期寫入筆數: 30
總實際寫入筆數: 30


In [2]:
# 統計特定詞彙在 JD requirement 中的出現次數與職缺數："NLP"、"自然語言處理"、"關聯式資料庫"

import os
import pandas as pd

DATA_DIR = os.getcwd()
JOBS_CSV = os.path.join(DATA_DIR, "jobs_rows.csv")

# 這裡以 job_description + other_requirements 當作 JD requirement 來源文字
cols = ["job_id", "job_description", "other_requirements"]
jobs_text = pd.read_csv(JOBS_CSV, usecols=cols, encoding="utf-8", low_memory=False)

# 合併兩個欄位成一個大文字欄位
jobs_text["jd_text"] = (
    jobs_text["job_description"].fillna("").astype(str)
    + "\n"
    + jobs_text["other_requirements"].fillna("").astype(str)
)

print(f"總職缺數: {len(jobs_text):,}")

terms = [
    ("NLP", True),          # 英文字母，使用不分大小寫統計
    ("自然語言處理", False),  # 精確中文字串
    ("關聯式資料庫", False),  # 精確中文字串
]

stats = []
for term, case_insensitive in terms:
    if case_insensitive:
        # 出現次數（多次出現在同一職缺也會累計）
        occ = jobs_text["jd_text"].str.lower().str.count(term.lower()).sum()
        # 至少出現一次的職缺數
        job_cnt = jobs_text["jd_text"].str.contains(term, case=False, na=False).sum()
    else:
        occ = jobs_text["jd_text"].str.count(term).sum()
        job_cnt = jobs_text["jd_text"].str.contains(term, case=False, na=False).sum()

    freq = job_cnt / len(jobs_text) if len(jobs_text) else 0.0
    stats.append({
        "term": term,
        "total_occurrences": int(occ),
        "job_count": int(job_cnt),
        "job_ratio": freq,
    })

print("=== 指定詞彙在 JD requirement 中的統計 ===")
for s in stats:
    print(
        f"{s['term']}: 總出現次數 = {s['total_occurrences']}, "
        f"出現於 {s['job_count']} 筆職缺 (約 {s['job_ratio']:.4f})"
    )

pd.DataFrame(stats)

總職缺數: 18,729
=== 指定詞彙在 JD requirement 中的統計 ===
NLP: 總出現次數 = 345, 出現於 246 筆職缺 (約 0.0131)
自然語言處理: 總出現次數 = 177, 出現於 145 筆職缺 (約 0.0077)
關聯式資料庫: 總出現次數 = 588, 出現於 528 筆職缺 (約 0.0282)


,term,total_occurrences,job_count,job_ratio
0,NLP,345,246,0.013135
1,自然語言處理,177,145,0.007742
2,關聯式資料庫,588,528,0.028192
